In [21]:
ls /media/notebooks/synthea/

avro/  encounters.csv  organizations.csv  parquet/  patients.csv


In [20]:
!hdfs dfs -mkdir -p /datasets/synthea/resultados
!hdfs dfs -mkdir -p /datasets/synthea/csv
!hdfs dfs -put  /media/notebooks/synthea/*.csv /datasets/synthea/csv

!hdfs dfs -ls /datasets/synthea/csv

Found 3 items
-rw-r--r--   3 root supergroup   78760575 2026-10-18 07:38 /datasets/synthea/csv/encounters.csv
-rw-r--r--   3 root supergroup       8356 2026-10-18 07:38 /datasets/synthea/csv/organizations.csv
-rw-r--r--   3 root supergroup    1123211 2026-10-18 07:38 /datasets/synthea/csv/patients.csv


In [22]:
%%writefile mapper_pacientes.py
#!/usr/bin/env python3

import sys
import csv

# Lectura del CSV desde la entrada estándar
reader = csv.DictReader(sys.stdin)

for row in reader:

    birthdate = row.get("BIRTHDATE", "").strip()
    gender = row.get("GENDER", "").strip()

    # Comprobar que existen los datos necesarios
    if not birthdate or not gender:
        continue

    try:
        # Obtener el año de nacimiento
        year = int(birthdate[:4])

        # Calcular la década
        decade = (year // 10) * 10

        # Generar la clave y el valor
        key = f"{decade}|{gender}"

        print(f"{key}\t1")

    except ValueError:
        continue

Overwriting mapper_pacientes.py


In [10]:
%%writefile reducer_pacientes.py
#!/usr/bin/env python3

import sys

current_key = None
total = 0

for line in sys.stdin:

    line = line.strip()

    if not line:
        continue

    try:
        key, value = line.split("\t", 1)
        value = int(value)
    except ValueError:
        continue

    # Primera clave
    if current_key is None:
        current_key = key
        total = value

    # Misma clave
    elif key == current_key:
        total += value

    # Nueva clave
    else:
        print(f"{current_key}\t{total}")

        current_key = key
        total = value

# Escribir la última clave
if current_key is not None:
    print(f"{current_key}\t{total}")

Writing reducer_pacientes.py


In [23]:
#1. Ejercicio — Pacientes por género y década

!hdfs dfs -rm -r -f /datasets/synthea/resultados/pacientes_genero_decada

!hadoop jar /usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
    -files mapper_pacientes.py,reducer_pacientes.py \
    -input /datasets/synthea/csv/patients.csv \
    -output /datasets/synthea/resultados/pacientes_genero_decada \
    -mapper "python3 mapper_pacientes.py" \
    -reducer "python3 reducer_pacientes.py"

packageJobJar: [/tmp/hadoop-unjar7067183478780933310/] [] /tmp/streamjob5622969943823734348.jar tmpDir=null
2026-10-18 07:39:08,630 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2026-10-18 07:39:08,913 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.3:8032
2026-10-18 07:39:09,480 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/root/.staging/job_1791539915649_0004
2026-10-18 07:39:11,299 INFO mapred.FileInputFormat: Total input files to process : 1
2026-10-18 07:39:11,717 INFO mapreduce.JobSubmitter: number of splits:2
2026-10-18 07:39:12,389 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1791539915649_0004
2026-10-18 07:39:12,389 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-10-18 07:39:13,992 INFO conf.Configuration: resource-types.xml not found
2026-10-18 07:39:13,993 INFO resource.ResourceUtils

In [24]:
!hdfs dfs -ls /datasets/synthea/resultados/pacientes_genero_decada

Found 2 items
-rw-r--r--   3 root supergroup          0 2026-10-18 07:40 /datasets/synthea/resultados/pacientes_genero_decada/_SUCCESS
-rw-r--r--   3 root supergroup        251 2026-10-18 07:40 /datasets/synthea/resultados/pacientes_genero_decada/part-00000


In [25]:
!hdfs dfs -cat /datasets/synthea/resultados/pacientes_genero_decada/part-00000


1910|F	18
1910|M	5
1920|F	22
1920|M	28
1930|F	15
1930|M	9
1940|F	55
1940|M	51
1950|F	107
1950|M	89
1960|F	166
1960|M	136
1970|F	153
1970|M	140
1980|F	111
1980|M	130
1990|F	132
1990|M	121
2000|F	127
2000|M	139
2010|F	111
2010|M	109
2020|F	58
2020|M	60
